In [ ]:
# Parameters — edit these before running
REPO_URL    = "https://github.com/Dev252001/ham10000-lesion-eval.git"
REPO_BRANCH = "main"  # set to another branch to test before merging
SEED        = 42
SPLIT_MODE  = "lesion"  # "image" is for the leakage comparison
N_EXAMPLES  = 3  # Grad-CAM examples per class per result type (correct/incorrect)

In [ ]:
# Setup: mount Drive, clone / checkout / pull repo, install extras
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, subprocess, sys, importlib
from pathlib import Path

DRIVE_DIR  = Path("/content/drive/MyDrive/ham10000_project")
REPO_DIR   = DRIVE_DIR / "ham10000-lesion-eval"
CKPT_DIR   = DRIVE_DIR / "checkpoints"
DATA_DIR   = Path("/content/ham10000_data")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

# Always put the freshly-pulled src first on the path
src_dir = str(REPO_DIR / "src")
if src_dir in sys.path:
    sys.path.remove(src_dir)
sys.path.insert(0, src_dir)

# Drop any previously imported skinlesion modules so the fresh code is used
for mod in list(sys.modules.keys()):
    if mod == "skinlesion" or mod.startswith("skinlesion."):
        del sys.modules[mod]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pandas", "scikit-learn", "matplotlib", "Pillow",
     "grad-cam", "kaggle"],
    check=True,
)

print("Setup complete. Repo HEAD:",
      subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"]).decode().strip())

In [ ]:
# Copy dataset from Drive cache
import shutil

DATA_CACHE = DRIVE_DIR / "ham10000_raw"
DATA_DIR   = Path("/content/ham10000_data")
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
shutil.copytree(str(DATA_CACHE), str(DATA_DIR))
print(f"Data at {DATA_DIR}")

In [ ]:
# Caveat
#
# Grad-CAM highlights regions with a high gradient-weighted activation for
# the chosen class at the chosen convolutional layer. What these figures
# are: qualitative, per-image visualisations, useful for comparing models
# on the same images. What they are not: evidence of which features the
# model actually relies on, of clinical validity, or of generalisation.
# A heatmap on the lesion does not show that the model reasons about
# lesion structure; a heatmap off the lesion is a reason to look closer,
# not a diagnosis of shortcut learning. Describe these figures as
# qualitative observations only.
print("Caveat acknowledged.")

In [ ]:
# Load metadata, reconstruct split, load saved predictions
import json
import pandas as pd
import torch
from skinlesion.data import load_metadata
from skinlesion.splits import make_split, split_manifest, manifest_sha256

df = load_metadata(DATA_DIR)
train_df, val_df, test_df = make_split(df, split_mode=SPLIT_MODE, seed=SEED)
manifest = split_manifest(train_df, val_df, test_df, split_mode=SPLIT_MODE, seed=SEED)
msha = manifest_sha256(manifest)

preds_dir = REPO_DIR / "outputs" / "predictions"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"manifest sha256: {msha[:12]}...  device: {device}")

In [ ]:
# Load image paths from metadata
image_paths = {row['image_id']: row['image_path'] for _, row in df.iterrows()}
print(f"Loaded {len(image_paths)} image paths.")

In [ ]:
# --- Grad-CAM for resnet18 ---
import torch
from skinlesion.models import build_model
from skinlesion.config import TrainConfig
from skinlesion.gradcam import make_gradcam_figure

model_name = "resnet18"
cfg_r18    = TrainConfig(model_name=model_name, split_mode=SPLIT_MODE, seed=SEED)

# Load predictions
pred_file_r18  = preds_dir / f"{model_name}_{SPLIT_MODE}_seed{SEED}.csv"
pred_df_r18    = pd.read_csv(pred_file_r18)

# Verify meta fingerprint before loading checkpoint
ckpt_r18      = CKPT_DIR / f"{model_name}_{SPLIT_MODE}_seed{SEED}_best.pt"
meta_r18_path = ckpt_r18.parent / (ckpt_r18.stem + "_meta.json")
meta_r18      = json.loads(meta_r18_path.read_text(encoding="utf-8"))
print(f"resnet18 meta fingerprint: {meta_r18['config_fingerprint'][:12]}...")

model_r18 = build_model(model_name, pretrained=False)
model_r18.load_state_dict(torch.load(str(ckpt_r18), map_location=device, weights_only=False))
model_r18.eval()
model_r18.to(device)

figures_dir = REPO_DIR / "outputs" / "figures"
for cls in ("mel", "bcc", "akiec"):
    make_gradcam_figure(
        model=model_r18, model_name=model_name,
        pred_df=pred_df_r18, cls=cls,
        n=N_EXAMPLES, seed=SEED,
        image_paths=image_paths, figures_dir=figures_dir,
        device=device, image_size=cfg_r18.image_size,
    )
print("resnet18 Grad-CAM figures saved.")

In [ ]:
# --- Grad-CAM for efficientnet_b0 ---
from skinlesion.gradcam import make_gradcam_figure

model_name = "efficientnet_b0"
cfg_eff    = TrainConfig(model_name=model_name, split_mode=SPLIT_MODE, seed=SEED)

pred_file_eff  = preds_dir / f"{model_name}_{SPLIT_MODE}_seed{SEED}.csv"
pred_df_eff    = pd.read_csv(pred_file_eff)

ckpt_eff      = CKPT_DIR / f"{model_name}_{SPLIT_MODE}_seed{SEED}_best.pt"
meta_eff_path = ckpt_eff.parent / (ckpt_eff.stem + "_meta.json")
meta_eff      = json.loads(meta_eff_path.read_text(encoding="utf-8"))
print(f"efficientnet_b0 meta fingerprint: {meta_eff['config_fingerprint'][:12]}...")

model_eff = build_model(model_name, pretrained=False)
model_eff.load_state_dict(torch.load(str(ckpt_eff), map_location=device, weights_only=False))
model_eff.eval()
model_eff.to(device)

for cls in ("mel", "bcc", "akiec"):
    make_gradcam_figure(
        model=model_eff, model_name=model_name,
        pred_df=pred_df_eff, cls=cls,
        n=N_EXAMPLES, seed=SEED,
        image_paths=image_paths, figures_dir=figures_dir,
        device=device, image_size=cfg_eff.image_size,
    )
print("efficientnet_b0 Grad-CAM figures saved.")

In [ ]:
# Package outputs and download
from skinlesion.export import make_outputs_zip

zip_path = DRIVE_DIR / "outputs_04.zip"
make_outputs_zip(REPO_DIR, zip_path)
from google.colab import files
files.download(str(zip_path))
print("Unzip at the root of your local clone, then commit.")